# Transformer Inference Basics

Understand the token generation pipeline, KV cache mechanics, and prefill vs decode phases through hands-on experiments.

In [ ]:
import sys
sys.path.insert(0, '../../..')
import torch
import torch.nn as nn
import time
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
class MinimalTransformerBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model))
    def forward(self, x, kv_cache=None):
        h = self.ln1(x)
        out, _ = self.attn(h, h, h)
        x = x + out
        x = x + self.mlp(self.ln2(x))
        return x

model = nn.Sequential(*[MinimalTransformerBlock() for _ in range(6)]).to(device)
print(f'Model: 6 layers, {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

In [ ]:
# Shape annotations through the forward pass
x = torch.randn(2, 128, 512, device=device)  # [batch=2, seq=128, d_model=512]
print(f'Input:  {list(x.shape)}  [batch, seq_len, d_model]')
for i, layer in enumerate(model):
    x = layer(x)
    if i == 0: print(f'After layer 0: {list(x.shape)}  [batch, seq_len, d_model]')
print(f'Output: {list(x.shape)}  [batch, seq_len, d_model]')

In [ ]:
# KV Cache growth during autoregressive generation
print('KV Cache Growth During Generation:')
print(f'{"Step":<6} {"Cached Tokens":<15} {"Cache Size (KB)":<15}')
print('-' * 40)
d_model, n_heads, head_dim = 512, 8, 64
for step in range(1, 11):
    seq_len = 10 + step  # prompt=10, generating token by token
    # KV cache: 2 (K,V) * layers * heads * head_dim * seq_len * bytes
    cache_bytes = 2 * 6 * n_heads * head_dim * seq_len * 4
    print(f'{step:<6} {seq_len:<15} {cache_bytes/1024:<15.1f}')

In [ ]:
# Prefill vs Decode timing
batch, prompt_len, d = 4, 256, 512
x_prefill = torch.randn(batch, prompt_len, d, device=device)
x_decode = torch.randn(batch, 1, d, device=device)

# Warmup
for _ in range(3): model(x_prefill); model(x_decode)
if device == 'cuda': torch.cuda.synchronize()

# Prefill
t0 = time.perf_counter()
for _ in range(20): model(x_prefill)
if device == 'cuda': torch.cuda.synchronize()
prefill_ms = (time.perf_counter() - t0) / 20 * 1000

# Decode
t0 = time.perf_counter()
for _ in range(20): model(x_decode)
if device == 'cuda': torch.cuda.synchronize()
decode_ms = (time.perf_counter() - t0) / 20 * 1000

print(f'Prefill ({prompt_len} tokens): {prefill_ms:.2f}ms  -> {prompt_len/prefill_ms*1000:.0f} tok/s')
print(f'Decode  (1 token):       {decode_ms:.2f}ms  -> {1/decode_ms*1000:.0f} tok/s')
print(f'\nPrefill processes {prompt_len}x more tokens but is only {prefill_ms/decode_ms:.1f}x slower')
print(f'Because prefill reuses weights across all tokens (high arithmetic intensity)')

In [ ]:
# Memory breakdown
param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
kv_cache_bytes = 2 * 6 * 8 * 64 * 2048 * batch * 4  # at seq=2048
activation_bytes = batch * 2048 * 512 * 4 * 6  # rough estimate

labels = ['Weights', 'KV Cache\n(seq=2048, b=4)', 'Activations']
sizes_mb = [param_bytes/1e6, kv_cache_bytes/1e6, activation_bytes/1e6]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, sizes_mb, color=['#4285f4', '#ea4335', '#34a853'])
for bar, s in zip(bars, sizes_mb): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{s:.0f}MB', ha='center')
ax.set_ylabel('Memory (MB)')
ax.set_title('Memory Breakdown: Weights vs KV Cache vs Activations')
plt.tight_layout()
plt.show()

In [ ]:
# Batch size effect on throughput
batch_sizes = [1, 2, 4, 8, 16]
throughputs = []
for b in batch_sizes:
    x = torch.randn(b, 1, 512, device=device)
    for _ in range(3): model(x)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(50): model(x)
    if device == 'cuda': torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / 50 * 1000
    throughputs.append(b / ms * 1000)

plt.figure(figsize=(8, 4))
plt.plot(batch_sizes, throughputs, 'o-', linewidth=2)
plt.xlabel('Batch Size')
plt.ylabel('Decode Throughput (tok/s)')
plt.title('Batch Size vs Decode Throughput')
plt.grid(True, alpha=0.3)
plt.show()
print(f'Scaling: batch 1={throughputs[0]:.0f} tok/s, batch 16={throughputs[-1]:.0f} tok/s ({throughputs[-1]/throughputs[0]:.1f}x)')

In [ ]:
# Generation timeline visualization
np.random.seed(42)
prompt_tokens = 50
gen_tokens = 30
prefill_time = 15  # ms
decode_times = np.random.exponential(3, gen_tokens) + 2  # ms per token

fig, ax = plt.subplots(figsize=(12, 3))
# Prefill bar
ax.barh(0, prefill_time, left=0, height=0.4, color='#4285f4', label=f'Prefill ({prompt_tokens} tok)')
# Decode bars
cum = prefill_time
for i, dt in enumerate(decode_times):
    ax.barh(0, dt, left=cum, height=0.4, color='#ea4335' if i==0 else '#fbbc04', alpha=0.7)
    cum += dt
ax.axvline(prefill_time, color='black', linestyle='--', alpha=0.5)
ax.text(prefill_time/2, 0.3, 'PREFILL', ha='center', fontsize=9)
ax.text(prefill_time + (cum-prefill_time)/2, 0.3, 'DECODE', ha='center', fontsize=9)
ax.set_xlabel('Time (ms)')
ax.set_title(f'Generation Timeline: {prompt_tokens} prompt + {gen_tokens} generated tokens')
ax.set_yticks([])
plt.tight_layout()
plt.show()

## Key Takeaways

| Insight | Implication |
|---------|-------------|
| Prefill is compute-bound | Benefits from more FLOPS (H100 > A100) |
| Decode is memory-bandwidth-bound | Benefits from more BW (HBM3 > HBM2e) |
| KV cache grows linearly with seq_len | Long context = memory pressure |
| Batching improves decode throughput | But increases KV cache proportionally |
| Weights are fixed cost, KV cache is variable | At scale, KV dominates memory |